# ElevenLabs TTS + 음향 교란 파일럿 (문장 1개)

CSV 첫 문장 1개 → ElevenLabs TTS(clean) → 장부(ledger.csv) 기록 → 교란 4개(reverb_room, reverb_large, echo, whisper) 생성 → 장부 갱신.

API 키는 환경변수 `ELEVENLABS_API_KEY`가 있으면 그 값을 쓰고, 없으면 실행할 때 입력창에서 받습니다. 키를 노트북 코드에 직접 적지 마세요.

CSV는 `advbench_ko.csv`의 `goal_ko` 열(한국어 번역)을 씁니다.

In [ ]:
# 패키지가 없을 때만 실행 (Colab이나 설치된 환경이면 건너뛰어도 됩니다)
!pip install -q requests soundfile librosa scipy pandas numpy

In [ ]:
import os
import getpass
import numpy as np
import pandas as pd
import requests
import soundfile as sf
import librosa
from scipy.signal import fftconvolve, butter, sosfilt
from IPython.display import Audio, display

SR = 16000
SEED = 42
WORK_DIR = 'work_test'
LEDGER_PATH = f'{WORK_DIR}/ledger.csv'
os.makedirs(f'{WORK_DIR}/audio', exist_ok=True)

CSV_PATH = 'advbench_ko.csv'
TEXT_COL = 'goal_ko'
VOICE_ID = 'EbiSUflQjzm4g7S895SC'
MODEL_ID = 'eleven_multilingual_v2'
PROMPT_ID = 'test_000'

API_KEY = os.environ.get('ELEVENLABS_API_KEY') or getpass.getpass('ElevenLabs API 키: ')

PERTS = ['reverb_room', 'reverb_large', 'echo', 'whisper']
COLS = ['prompt_id', 'text', 'clean'] + PERTS

## 1. CSV에서 첫 문장 가져오기

In [ ]:
df = pd.read_csv(CSV_PATH)
print('열 이름:', df.columns.tolist())
TEXT = str(df.iloc[0][TEXT_COL])
print('테스트 문장:', TEXT)

## 2. 장부(ledger) 준비

장부는 항목마다 clean과 교란 4개의 상태(대기/완료/실패)를 기록합니다. 파일을 쓴 뒤에 상태를 갱신합니다.

In [ ]:
def load_ledger():
    if os.path.exists(LEDGER_PATH):
        return pd.read_csv(LEDGER_PATH, dtype=str).fillna('')
    return pd.DataFrame(columns=COLS)

def save_ledger(ld):
    ld.to_csv(LEDGER_PATH, index=False, encoding='utf-8-sig')

def set_status(ld, pid, col, val):
    ld.loc[ld.prompt_id == pid, col] = val
    save_ledger(ld)

ledger = load_ledger()
if PROMPT_ID not in ledger.prompt_id.values:
    new_row = {'prompt_id': PROMPT_ID, 'text': TEXT, 'clean': '대기'}
    for p in PERTS:
        new_row[p] = '대기'
    ledger = pd.concat([ledger, pd.DataFrame([new_row])], ignore_index=True)
    save_ledger(ledger)
display(ledger)

## 3. ElevenLabs TTS → clean wav

`output_format=pcm_16000`으로 받으면 16kHz 모노 PCM이 바로 오므로, int16을 float로 바꿔 wav로 저장합니다.

In [ ]:
def tts_clean(text, pid):
    url = f'https://api.elevenlabs.io/v1/text-to-speech/{VOICE_ID}'
    headers = {'xi-api-key': API_KEY, 'Content-Type': 'application/json'}
    body = {
        'text': text,
        'model_id': MODEL_ID,
        'voice_settings': {'stability': 0.0, 'similarity_boost': 0.5},
    }
    r = requests.post(url, params={'output_format': 'pcm_16000'}, headers=headers, json=body, timeout=120)
    r.raise_for_status()
    pcm = np.frombuffer(r.content, dtype=np.int16).astype(np.float32) / 32768.0
    out = f'{WORK_DIR}/audio/clean/{pid}.wav'
    os.makedirs(os.path.dirname(out), exist_ok=True)
    sf.write(out, pcm, SR)
    return out

ledger = load_ledger()
status = ledger.loc[ledger.prompt_id == PROMPT_ID, 'clean'].iloc[0]
if status != '완료':
    try:
        clean_path = tts_clean(TEXT, PROMPT_ID)
        set_status(ledger, PROMPT_ID, 'clean', '완료')
        print('clean 저장:', clean_path)
    except Exception as e:
        set_status(ledger, PROMPT_ID, 'clean', f'실패: {e}')
        raise
else:
    clean_path = f'{WORK_DIR}/audio/clean/{PROMPT_ID}.wav'
    print('이미 완료됨, 건너뜀:', clean_path)

## 4. 교란 4개 생성

IR 파일이 없으므로 합성 IR을 씁니다. 결과는 조건 간 상대 비교용입니다.

In [ ]:
def peak_normalize(x):
    m = np.max(np.abs(x))
    return (x / m).astype(np.float32) if m > 0 else x.astype(np.float32)

def synthetic_ir(rt60, drr_db, seed=0):
    rng = np.random.default_rng(seed)
    n = int(SR * rt60 * 1.2)
    t = np.arange(n) / SR
    ir = rng.standard_normal(n) * np.exp(-6.908 * t / rt60)
    ir[0] = 0.0
    ir *= np.sqrt(10 ** (-drr_db / 10) / np.sum(ir ** 2))
    ir[0] = 1.0
    return ir.astype(np.float32)

IRS = {
    'reverb_room': synthetic_ir(rt60=0.6, drr_db=0.0),
    'reverb_large': synthetic_ir(rt60=2.0, drr_db=-5.0),
}

def add_echo(x, delay=0.3, decay=0.6):
    d = int(SR * delay)
    y = np.zeros(len(x) + d, dtype=np.float32)
    y[:len(x)] += x
    y[d:] += decay * x
    return peak_normalize(y)

_WHISPER_SOS = butter(4, 1500, btype='low', fs=SR, output='sos')

def simulate_whisper(x, reduction=0.3, noise_level=0.005, seed=0):
    rng = np.random.default_rng(seed)
    y = sosfilt(_WHISPER_SOS, x * reduction)
    y = y + noise_level * rng.standard_normal(len(y))
    return peak_normalize(y)

def perturb(x, name, seed):
    if name in IRS:
        return peak_normalize(fftconvolve(x, IRS[name], mode='full'))
    if name == 'echo':
        return add_echo(x)
    if name == 'whisper':
        return simulate_whisper(x, seed=seed)
    raise ValueError(name)

ledger = load_ledger()
x = librosa.load(clean_path, sr=SR, mono=True)[0].astype(np.float32)
for p in PERTS:
    if ledger.loc[ledger.prompt_id == PROMPT_ID, p].iloc[0] == '완료':
        print(p, '이미 완료됨, 건너뜀')
        continue
    try:
        out = f'{WORK_DIR}/audio/{p}/{PROMPT_ID}.wav'
        os.makedirs(os.path.dirname(out), exist_ok=True)
        sf.write(out, perturb(x, p, SEED), SR)
        set_status(ledger, PROMPT_ID, p, '완료')
        print(p, '저장:', out)
    except Exception as e:
        set_status(ledger, PROMPT_ID, p, f'실패: {e}')
        print(p, '실패:', e)

## 5. 결과 확인 (장부 + 듣기)

In [ ]:
display(load_ledger())
for p in ['clean'] + PERTS:
    path = f'{WORK_DIR}/audio/{p}/{PROMPT_ID}.wav'
    if os.path.exists(path):
        print(p)
        display(Audio(path))

In [ ]:
import sys; print(sys.executable)


In [ ]:
!"c:\ProgramData\anaconda3\python.exe" -m pip install -q soundfile librosa scipy pandas requests numpy
